# Proy 1

In [45]:
import numpy as np
from scipy.stats import erlang
from scipy.integrate import quad
import scipy.optimize as opt

In [46]:
So = 19.90 # Precio de referencia
K = 60
Lam = 3.0

pi_i = 0.4 # Prob. informado
pi_l = 0.6 # Prob liquidez


In [47]:
#Definimos función erlang
def f(P):
    return erlang.pdf(P, K, scale=1/Lam)

def prob_ejecucion(s):
    return max(0.50 - 0.08 * s, 0)

In [48]:
# Definimos funciones de ganancia y pérdida para el modelo de liquidez e informados
def ganancia_liquidez(A, B):
    s_ask = A - So
    s_bid = So - B
    return prob_ejecucion(s_ask) * (A - So) + prob_ejecucion(s_bid) * (So - B)

def perdida_informados(A, B):
    perdida_compra, _ = quad(lambda P: (P - A) * f(P), A, np.inf)
    perdida_venta,  _ = quad(lambda P: (B - P) * f(P), 0.0, B)
    return perdida_compra + perdida_venta

def Pi(A, B):
    return pi_l * ganancia_liquidez(A, B) - pi_i * perdida_informados(A, B)

def objetivo(x):
    A, B = x
    return -Pi(A, B)

In [49]:

bounds = [(So, None), (1e-6, So)]        # A ∈ [So, ∞), B ∈ (0, So] restricciónes sacadas de informe
x0 = [So + 0.10, So - 0.10]              

resultado = opt.minimize(objetivo, x0=x0, bounds=bounds) #optimización

A_opt, B_opt = resultado.x


In [50]:
#para calcular el spread usamos el roll in window APLICARLO Y CORREGIRLO
spread   = A_opt - B_opt
utilidad = Pi(A_opt, B_opt)


In [51]:

print(f"Ask óptimo (A):    {A_opt:.2f}")
print(f"Bid óptimo (B):    {B_opt:.2f}")
print(f"Spread:            {spread:.2f}")
print(f"Utilidad esperada: {utilidad:.2f}")

Ask óptimo (A):    23.43
Bid óptimo (B):    16.45
Spread:            6.98
Utilidad esperada: 0.84


## Analisis de sensibilidad 

In [52]:
#Creamos tabla con los resultados 

data = pd.DataFrame({
    'Ask óptimo (A)': [A_opt],
    'Bid óptimo (B)': [B_opt],
    'Spread óptimo (S)': [spread],
    'Utilidad esperada (U)': [utilidad]
})
data

,Ask óptimo (A),Bid óptimo (B),Spread óptimo (S),Utilidad esperada (U)
0,23.427664,16.45168,6.975984,0.840285


In [53]:
# Creamos un analisis de sensibilidad con los valores para calcular +10% y -10%
spread_ask_opt = A_opt - So
spread_bid_opt = B_opt - So

Spread_ask_mas10 = So + spread_ask_opt * 1.10
Spread_bid_mas10 = So - spread_bid_opt * 1.10

Spread_ask_menos10 = So + spread_ask_opt * 0.90
Spread_bid_menos10 = So - spread_bid_opt * 0.90

regimenes["Óptimo +10%"] = (Spread_ask_mas10, Spread_bid_mas10)
regimenes["Óptimo -10%"] = (Spread_ask_menos10, Spread_bid_menos10)


for nombre in ["Óptimo", "Óptimo +10%", "Óptimo -10%"]:
    A, B = regimenes[nombre]
    print(f"{nombre}: A={A:.2f}, B={B:.2f}, spread={A-B:.2f}, Pi teórica={Pi(A,B):.4f}")

print()

for nombre in ["Óptimo", "Óptimo +10%", "Óptimo -10%"]:
    A, B = regimenes[nombre]
    pnl_finales = simular_corridas(A, B, 1_000, 1_000, seed=7)
    print(f"{nombre:14s} | P&L prom={pnl_finales.mean():10.2f} | std={pnl_finales.std():8.2f} | "
          f"P(pérdida)={np.mean(pnl_finales < 0):.4f}")

Óptimo: A=23.43, B=16.45, spread=6.98, Pi teórica=0.8403
Óptimo +10%: A=23.78, B=23.69, spread=0.09, Pi teórica=-2.9506
Óptimo -10%: A=23.07, B=23.00, spread=0.07, Pi teórica=-2.2678

Óptimo         | P&L prom=   2511.93 | std=   64.33 | P(pérdida)=0.0000
Óptimo +10%    | P&L prom=  -2969.14 | std=   91.34 | P(pérdida)=1.0000
Óptimo -10%    | P&L prom=  -2282.32 | std=   84.46 | P(pérdida)=1.0000


## Analisis Montecarlo